In [1]:
# Import required libraries
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import csv
from torchvision import models
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from sklearn.metrics import precision_recall_fscore_support
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

# os.environ["TORCH_HOME"] = '/opt/torch_models'

# Constants
NUM_CLASSES = 100
TARGET_FRAMES = 16  # number of frames per video

In [2]:
# Read video frames using OpenCV
def read_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()

        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    cap.release()
    if len(frames) == 0:
        raise ValueError(f"Could not read any frames from {video_path}")
    frames = torch.from_numpy(np.stack(frames, axis=0))
    return frames


# Custom collate function for batching
def collate_fn(batch):
    frames = torch.stack([item['frames'] for item in batch])
    labels = torch.tensor([item['label_idx'] for item in batch])
    label_names = [item['label'] for item in batch]
    return {'frames': frames, 'label_idx': labels, 'label': label_names}

In [3]:
# Define video dataset
class VideoDataset(Dataset):
    def __init__(self, root_dir, label_to_idx_path, transform=None,
                 mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225],
                 target_frames=32):
        self.root_dir = root_dir
        self.transform = transform
        self.mean, self.std = mean, std
        self.target_frames = target_frames
        self.instances, self.labels, self.label_idx = [], [], []

        with open(label_to_idx_path, 'rb') as f:
            self.label_mapping = pickle.load(f)

        for label_folder in sorted(os.listdir(root_dir))[:NUM_CLASSES]:
            path = os.path.join(root_dir, label_folder)
            if os.path.isdir(path):
                for video_file in os.listdir(path):
                    video_path = os.path.join(path, video_file)
                    self.instances.append(video_path)
                    self.labels.append(label_folder)
                    self.label_idx.append(self.label_mapping[label_folder])

    # Downsample frames to fixed length
    def _downsample_frames(self, frames):
        num_frames = frames.shape[0]
        if num_frames == self.target_frames:
            return frames
        elif num_frames < self.target_frames:
            pad = self.target_frames - num_frames
            return torch.cat([frames, frames[-1:].repeat(pad, 1, 1, 1)], dim=0)
        else:
            idx = torch.linspace(0, num_frames - 1, self.target_frames).long()
            return frames[idx]

    # Normalize frames with ImageNet stats
    def _normalize(self, frames):
        frames = frames.permute(0, 3, 1, 2).float() / 255.0
        mean = torch.tensor(self.mean).view(1, 3, 1, 1)
        std = torch.tensor(self.std).view(1, 3, 1, 1)
        return (frames - mean) / std

    def __len__(self):
        return len(self.instances)

    def __getitem__(self, idx):
        video_path = self.instances[idx]
        label, label_idx = self.labels[idx], self.label_idx[idx]
        frames = read_video(video_path)
        frames = self._downsample_frames(frames)
        frames = self._normalize(frames)
        return {"frames": frames, "label_idx": label_idx, "label": label}

In [4]:
# Define CRNN model
class CRNN(nn.Module):
    def __init__(self, num_classes=100, hidden_size=256, resnet_pretrained_weights=None):
        super(CRNN, self).__init__()
        resnet = models.resnet18(weights=resnet_pretrained_weights)
        self.cnn = nn.Sequential(*list(resnet.children())[:-2])
        self.feature_dim = 512
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        # code cu ===
        # self.rnn = nn.LSTM(self.feature_dim, hidden_size, batch_first=True, dropout=0.3)
        # self.fc = nn.Linear(hidden_size, num_classes)
        # code moi bat bidirection va tang them 1 layer, sua lai input size do bat hai chieu, so luong chieu * 2
        self.rnn = nn.LSTM(self.feature_dim, hidden_size, batch_first=True, dropout=0.3, bidirectional=True, num_layers=2)
        self.fc = nn.Linear(hidden_size*4, num_classes) # hidden size * 4 la do cac thao tac o duoi 

    def forward(self, x):
        B, T, C, H, W = x.size()
        x = x.view(B * T, C, H, W)
        features = self.cnn(x)
        pooled = self.pool(features).squeeze(-1).squeeze(-1)
        seq = pooled.view(B, T, self.feature_dim) # seq.shape = (B, T, N)
        rnn_out, _ = self.rnn(seq) # rnn.shape = (B, T, N*2)
        # code cu 
        # final = rnn_out[:, -1, :]
        # code moi (them mean + max pooling )
        # mean_pool va max_pool co shape = (B, N*2)
        mean_pool = torch.mean(rnn_out, dim=1)
        max_pool = torch.mean(rnn_out, dim=1)
        final = torch.cat([mean_pool, max_pool], dim=1) # final.shape = (B, N*4) => sua len cai self.fc  
        
        return self.fc(final)

In [5]:
# One training epoch
def train_epoch(model, dataloader, criterion, optimizer, device='cuda'):
    model.train()
    total_loss = 0
    progress = tqdm(dataloader, desc='Training')
    for batch in progress:
        frames, labels = batch['frames'].to(device, non_blocking=True), batch['label_idx'].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        # them automatic mixed precision 
        with torch.autocast(device, torch.float16):
            outputs = model(frames)
            loss = criterion(outputs, labels)
        # code cu ====
        # loss.backward()
        # optimizer.step()
        
        # cai tien them AMP va gradient clipping ===
        scaler.scaler(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update
        # ============
        
        total_loss += loss.item()
        progress.set_postfix({'loss': f'{total_loss / (len(progress)+1e-9):.4f}'})
    return total_loss / len(dataloader)


# Validation
def validate(model, dataloader, criterion, device='cuda'):
    model.eval()
    total_loss, preds, labels_all = 0, [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Validation'):
            frames, labels = batch['frames'].to(device, non_blocking=True), batch['label_idx'].to(device, non_blocking=True)
            outputs = model(frames)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            preds.extend(predicted.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    precision, recall, f1, _ = precision_recall_fscore_support(labels_all, preds, average='macro', zero_division=0)
    return total_loss / len(dataloader), {'precision': precision*100, 'recall': recall*100, 'f1': f1*100}

In [6]:
weights = torch.ones(NUM_CLASSES)
for i in range(65, 100):
    weights[i] += 0.5
weights

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
        1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
        1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
        1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
        1.5000])

In [7]:
# Full training loop with validation and test evaluation
def train_model(model, train_loader, val_loader,
                num_epochs=10, lr=5e-4, device='cuda', save_path='best_model.pth'):
    model = model.to(device, non_blocking=True)
    # cai tien 
    model = torch.compile(model)
    # ========
    # code cu ===
    # criterion = nn.CrossEntropyLoss()
    # code moi them label smoothing va weight
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1, weight=weights)
    # ===
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=3)

    best_f1 = 0.0
    ## them AMP
    scaler = torch.amp.GradScaler(device)
    for epoch in range(num_epochs):
        print(f"\n===== Epoch {epoch+1}/{num_epochs} =====")
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_metrics = validate(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        print(f"Val F1: {val_metrics['f1']:.2f}% | Precision: {val_metrics['precision']:.2f}% | Recall: {val_metrics['recall']:.2f}%")

        if val_metrics['f1'] > best_f1:
            best_f1 = val_metrics['f1']
            torch.save(model.state_dict(), save_path)
            print(f"✓ Best model saved with F1: {best_f1:.2f}%")
    return model

In [8]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
from torchvision.transforms import v2 
from torchvision.models import ResNet18_Weights

In [9]:
# Training
model = CRNN(num_classes=NUM_CLASSES, hidden_size=256,
             resnet_pretrained_weights=models.ResNet18_Weights.IMAGENET1K_V1)

# code cu
# full_train = VideoDataset('dataset/train', 'dataset/label_mapping.pkl', target_frames=TARGET_FRAMES)
# train_size = int(0.8 * len(full_train))
# val_size = len(full_train) - train_size
# train_dataset, val_dataset = random_split(full_train, [train_size, val_size], generator=torch.Generator().manual_seed(42))
# code moi (them data augmentation)
train_transform = v2.Compose([
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0)
])
val_transform = None

full_train = VideoDataset('dataset/train', 'dataset/label_mapping.pkl', target_frames=TARGET_FRAMES)
full_val = VideoDataset('dataset/train', 'dataset/label_mapping.pkl', target_frames=TARGET_FRAMES)

train_indices, val_indices = train_test_split(torch.arange(len(full_train.labels)), lengths=[0.8, 0.2], random_state=42, stratify=full_train.labels_idx)
train_dataset = Subset(full_train, train_indices)
val_dataset = Subset(full_val, val_indices)

# cai tien DataLoader 
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn, num_workers=128, pin_memory=True, persistent_workers=True, prefetch_factor=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=128, pin_memory=True, persistent_workers=True, prefetch_factor=2)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# model = train_model(model, train_loader, val_loader, num_epochs=20, lr=1e-4, device='cuda', save_path='best_model.pth')

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/label_mapping.pkl'

In [ ]:
# Evaluate trained model on test set
def evaluate(model, folder_path, label_to_idx_path, output_csv="predictions.csv",
             device='cuda', model_path=None, target_frames=16):
    # Load trained weights if provided
    if model_path:
        model.load_state_dict(torch.load(model_path))
        print(f"Loaded model from {model_path}")

    model = model.to(device, non_blocking=True)
    model.eval()

    # Load label mapping
    with open(label_to_idx_path, 'rb') as f:
        label_mapping = pickle.load(f)
    idx_to_label = {v: k for k, v in label_mapping.items()}

    # Collect video files
    video_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))])
    print(f"Found {len(video_files)} videos in '{folder_path}'")

    predictions = []

    dataset = VideoDataset(
                  root_dir=folder_path,
                  label_to_idx_path=label_to_idx_path,
                  target_frames=target_frames
              )

    with torch.no_grad():
        for video_file in tqdm(video_files, desc="Predicting"):
            video_path = os.path.join(folder_path, video_file)
            try:
                # Read and preprocess video
                frames = read_video(video_path)
                frames = dataset._downsample_frames(frames)
                frames = dataset._normalize(frames)
                frames = frames.unsqueeze(0).to(device, non_blocking=True)  # (1, T, C, H, W)

                # Predict
                outputs = model(frames)
                _, predicted = outputs.max(1)
                label_idx = predicted.item()
                label_name = idx_to_label[label_idx]

                predictions.append((video_file, label_name))
            except Exception as e:
                print(f"Error processing {video_file}: {e}")

    # Save to CSV
    with open(output_csv, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['video_name', 'label'])
        writer.writerows(predictions)

    print(f"\nPredictions saved to '{output_csv}'")
    print(f"Total videos processed: {len(predictions)}")

In [ ]:
# Export public result
evaluate(
    model=model,
    folder_path="dataset/public_test",
    label_to_idx_path="dataset/label_mapping.pkl",
    model_path="best_model.pth",
    output_csv="public_submission.csv",
    device="cuda",
    target_frames=16
)

import zipfile

with zipfile.ZipFile("public_submission.zip", 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write("public_submission.csv")
        print("Created file public_submission.zip successfully.")

Loaded model from best_model.pth
Found 1630 videos in 'dataset/public_test'


Predicting: 100%|██████████| 1630/1630 [02:42<00:00, 10.00it/s]



Predictions saved to 'public_submission.csv'
Total videos processed: 1630
Created file public_submission.zip successfully.


In [ ]:
# Export private result
evaluate(
    model=model,
    folder_path="dataset/private_test",
    label_to_idx_path="dataset/label_mapping.pkl",
    model_path="best_model.pth",
    output_csv="private_submission.csv",
    device="cuda",
    target_frames=16
)

import zipfile

with zipfile.ZipFile("private_test.zip", 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write("private_submission.csv")
        print("Created file private_submission.zip successfully.")